# Phase 6: Hyperparameter Optimization (Finding the Sweet Spot)

In a professional portfolio, it is critical to prove you didn't just 'guess' the model settings. 

We will use **RandomizedSearchCV** to mathematically test dozens of combinations for our XGBoost engine. We will use a 1-Million row sample for speed, then apply the winning settings to our 13.9M row production model.

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
import time

print("Loading 1-Million Row Representative Sample for Tuning...")
# Load only first 1M rows for efficiency in the laboratory
X_train = pd.read_csv("../data/X_train.csv", nrows=1000000)
y_train = pd.read_csv("../data/y_train.csv", nrows=1000000)

# We only tune the 'Treatment' model as it is the most critical for Uplift
mask_treat = X_train['treatment'] == 1
X_tune = X_train[mask_treat].drop(columns=['treatment'])
y_tune = y_train[mask_treat]
print(f"Tuning Sandbox Ready with {len(X_tune):,} treated users.")

---
## Step 1: Defining the Search Grid
We tell Python exactly which dials we want to turn.

In [ ]:
param_grid = {
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

base_model = XGBClassifier(
    tree_method='hist',
    device='cuda', # Use GPU if available
    n_jobs=-1
)

print("Search Grid Defined. Preparing to test combinations...")

---
## Step 2: Running the Search
We will test 10 random combinations using 3-Fold Cross-Validation. This means we rotate the data 3 times to ensure the results are robust and not based on 'luck' (another massive Portfolio Flex).

In [ ]:
search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_grid,
    n_iter=10, # Number of random settings to try
    scoring='roc_auc', # We want the model that ranks buyers best
    cv=3, # 3-Fold Cross-Validation
    verbose=2,
    random_state=42
)

start_time = time.time()
print(f"Searching for the Winning Parameters...")
search.fit(X_tune, y_tune)
end_time = time.time()

print(f"\n✅ Search Completed in {(end_time - start_time):.2f} seconds.")
print("\n--- THE WINNING HYPERPARAMETERS ---")
print(search.best_params_)
print(f"\nBest Validation ROC-AUC: {search.best_score_:.4f}")

---
## Step 3: Next Step - Applying the Winners
Now that we have discovered the 'Best Params', we must update our production script (`train_full_model.py`) to use these exact settings for the final 13.9M row run!